# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [17]:
!pip install duckdb --quiet

import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('HF_TOKEN')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

In [18]:
con.sql(f"""
    SELECT COUNT(*), MIN(report_date), MAX(report_date)
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┬──────────────────┬──────────────────┐
│ count_star() │ min(report_date) │ max(report_date) │
│    int64     │       date       │       date       │
├──────────────┼──────────────────┼──────────────────┤
│     10424730 │ 2026-04-01       │ 2026-04-30       │
└──────────────┴──────────────────┴──────────────────┘

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Logistic regression. Readable and a suitable next step after my decision tree in notebook 2.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: grouped by client, not by row.

The same client's pages would otherwise appear in both train and test,
letting the model learn "this specific client tends to decline" instead
of a genuinely generalizable pattern -- the same leakage risk flagged in
the flyrank-data skill. Splitting by client_hash_id first, then assigning
all of that client's pages to one side, keeps train and test genuinely
separate.

In [19]:
# March: one row per page, summarizing the month's behavior
march_agg = con.sql(f"""
    SELECT
        f.client_hash_id, f.content_hash_id,
        SUM(f.gsc_impressions) AS march_impressions,
        SUM(f.gsc_clicks) AS march_clicks,
        AVG(f.gsc_avg_position) AS march_avg_position,
        MAX(CASE
            WHEN DATE_DIFF('day', c.content_updated_date, f.report_date) < 0 THEN NULL
            ELSE DATE_DIFF('day', c.content_updated_date, f.report_date)
        END) AS days_since_update
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet') f
    JOIN read_parquet('{rel}/dim_content.parquet') c
        ON f.content_hash_id = c.content_hash_id AND f.client_hash_id = c.client_hash_id
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

# April: same aggregation, kept separate -- only ever used to build the label, never as a feature
april_agg = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

from sklearn.model_selection import train_test_split

unique_clients = march_agg['client_hash_id'].unique()
train_clients, test_clients = train_test_split(unique_clients, test_size=0.3, random_state=42)

print(f"March pages: {len(march_agg)}, April pages: {len(april_agg)}")
print(f"Train clients: {len(train_clients)}, Test clients: {len(test_clients)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March pages: 176738, April pages: 194760
Train clients: 32, Test clients: 15


In [20]:
# Merge March behavior with April outcome. left join -- a page that vanished
# from April entirely gets treated as 0 April clicks (arguably the strongest
# form of decline: it dropped off completely)
labeled = march_agg.merge(april_agg, on=['client_hash_id', 'content_hash_id'], how='left')
labeled['april_clicks'] = labeled['april_clicks'].fillna(0)

# Label: did clicks meaningfully drop from March to April?
# Using a 20% drop threshold, not "any decrease", to avoid labeling normal
# day-to-day noise as "declining"
labeled['declining'] = (labeled['april_clicks'] < labeled['march_clicks'] * 0.8).astype(int)

print(f"Total pages: {len(labeled)}")
print(f"Base rate (declining): {labeled['declining'].mean():.3f}")

Total pages: 176738
Base rate (declining): 0.237


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [21]:
# Split the labeled data by the client groups from Section 2
train_df = labeled[labeled['client_hash_id'].isin(train_clients)].copy()
test_df = labeled[labeled['client_hash_id'].isin(test_clients)].copy()

print(f"Train pages: {len(train_df)}, Test pages: {len(test_df)}")
print(f"Train base rate: {train_df['declining'].mean():.3f}, Test base rate: {test_df['declining'].mean():.3f}")

# Features: safe, March-only signals -- same shape as ML-04's feature classification
features = ['march_impressions', 'march_clicks', 'march_avg_position', 'days_since_update']

# days_since_update has real missing values (per ML-07's leakage fix) --
# fill with a large number so "unknown staleness" doesn't look artificially fresh
train_df['days_since_update'] = train_df['days_since_update'].fillna(9999)
test_df['days_since_update'] = test_df['days_since_update'].fillna(9999)

X_train, y_train = train_df[features], train_df['declining']
X_test, y_test = test_df[features], test_df['declining']

Train pages: 111419, Test pages: 65319
Train base rate: 0.260, Test base rate: 0.197


In [22]:
from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Model's predicted probability of decline -- this is what ranks the test set
model_scores = model.predict_proba(X_test)[:, 1]

# Baseline replica: same rule as ML-07 (staleness + CTR-underperformance + visibility),
# applied to the SAME aggregated March data, so the comparison is fair
test_df['ctr'] = np.where(test_df['march_impressions'] > 0,
                            test_df['march_clicks'] / test_df['march_impressions'], 0)
expected_ctr = np.where(test_df['march_avg_position'] <= 10, 0.0039, 0.0018)
ctr_underperforming = test_df['ctr'] < (expected_ctr * 0.5)
is_stale = test_df['days_since_update'] >= 180
visible = test_df['march_impressions'] >= 10

baseline_scores = np.select(
    [is_stale & visible & ctr_underperforming, is_stale & visible, ctr_underperforming & visible],
    [test_df['march_impressions'], test_df['march_impressions'] * 0.5, test_df['march_impressions'] * 0.3],
    default=0
)

# precision@k, same function shape as notebook 02
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
print(f"Base rate: {base_rate:.3f}\n")
for k in (20, 50):
    print(f"Precision@{k}:  baseline {precision_at_k(baseline_scores, y_test, k):.3f}   vs   model {precision_at_k(model_scores, y_test, k):.3f}")

Base rate: 0.197

Precision@20:  baseline 0.400   vs   model 0.500
Precision@50:  baseline 0.480   vs   model 0.380


In [23]:
import numpy as np

from sklearn.linear_model import LogisticRegression

# Train the model
model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

# Model's predicted probability of decline -- this is what ranks the test set
model_scores = model.predict_proba(X_test)[:, 1]

# Baseline replica: same rule as ML-07 (staleness + CTR-underperformance + visibility),
# applied to the SAME aggregated March data, so the comparison is fair
test_df['ctr'] = np.where(test_df['march_impressions'] > 0,
                            test_df['march_clicks'] / test_df['march_impressions'], 0)
expected_ctr = np.where(test_df['march_avg_position'] <= 10, 0.0039, 0.0018)
ctr_underperforming = test_df['ctr'] < (expected_ctr * 0.5)
is_stale = test_df['days_since_update'] >= 180
visible = test_df['march_impressions'] >= 10

baseline_scores = np.select(
    [is_stale & visible & ctr_underperforming, is_stale & visible, ctr_underperforming & visible],
    [test_df['march_impressions'], test_df['march_impressions'] * 0.5, test_df['march_impressions'] * 0.3],
    default=0
)

# precision@k, same function shape as notebook 02
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

base_rate = y_test.mean()
print(f"Base rate: {base_rate:.3f}\n")
for k in (20, 50):
    print(f"Precision@{k}:  baseline {precision_at_k(baseline_scores, y_test, k):.3f}   vs   model {precision_at_k(model_scores, y_test, k):.3f}")

Base rate: 0.197

Precision@20:  baseline 0.400   vs   model 0.500
Precision@50:  baseline 0.480   vs   model 0.380


Precision@20    Precision@50
Baseline        0.400           0.480
Model           0.500           0.380
Base rate       0.197           0.197

(Corrected after removing a leaked staleness signal -- see note below)

Both methods clear the base rate comfortably at both K values. The
crossover pattern from the uncorrected run held after the fix: the model
is stronger at the top 20 (0.500 vs 0.400), the baseline is stronger at
the top 50 (0.480 vs 0.380). This is a real, robust finding, not an
artifact of the leak -- fixing the leak shifted both numbers but didn't
change which method wins where.

Note: an earlier version of this notebook's March aggregation query
recalculated days_since_update without the leak guard built in ML-07 --
content_updated_date sometimes falls after report_date, which would leak
future information into a point-in-time feature. Caught during error
review (3 of the model's most confident wrong predictions all showed
negative days_since_update), fixed by nulling those rows instead of
treating them as a real staleness value.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [26]:
import pandas as pd
coef_df = pd.DataFrame({'feature': features, 'coefficient': model.coef_[0]})
coef_df.sort_values('coefficient', ascending=False)

,feature,coefficient
1,march_clicks,0.005089
0,march_impressions,0.000099
3,days_since_update,0.000013
2,march_avg_position,-0.020179


In [27]:
test_df['model_score'] = model_scores
test_df['actual'] = y_test.values

wrong_confident = test_df[(test_df['model_score'] > 0.5) & (test_df['actual'] == 0)].sort_values('model_score', ascending=False).head(3)
wrong_confident[['client_hash_id', 'content_hash_id', 'march_clicks', 'march_avg_position', 'days_since_update', 'model_score', 'actual']]

,client_hash_id,content_hash_id,march_clicks,march_avg_position,days_since_update,model_score,actual
94404,client_08a6a72ff48e62c0,content_e7b5dd4dff461ad2,2446.0,4.544203,9999,1.0,0
90666,client_73cda7b4e4f265ea,content_471d9cabce329a66,396.0,4.656030,9999,1.0,0
128084,client_73cda7b4e4f265ea,content_e241d6415ac9e534,343.0,3.276016,9999,1.0,0


Concrete wrong cases (model's most confident wrong predictions, post-fix):

Three test pages the model scored at ~1.0 probability of declining, but
actually didn't:
- client_08a6a72ff48e62c0 / content_e7b5dd4dff461ad2: 2,446 March clicks,
  position 4.5, staleness unknown (9999)
- client_73cda7b4e4f265ea / content_471d9cabce329a66: 396 March clicks,
  position 4.7, staleness unknown (9999)
- client_73cda7b4e4f265ea / content_e241d6415ac9e534: 343 March clicks,
  position 3.3, staleness unknown (9999)

All three share two traits: strong March performance (good position,
real click volume) and unknown staleness. This matches the regression-
to-the-mean explanation from the coefficient analysis -- pages doing
well in March have more room to drop 20%+ by April than pages already
struggling, so the model may be partly learning "successful pages
regress" rather than a genuine decline signal specific to these pages.
These are hard cases because nothing in the available March-only
features distinguishes "will regress toward the mean" from "will keep
performing well" -- that would need a longer history than one month.**bold text**

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.